# DualReform Phase 1 — bounded baseline audit

**GPU-backed results: NOT EXECUTED in the delivered notebook.** This notebook runs real-data integrity checks by default. It is not an official DualReform baseline. The official repository is unavailable; the accessible third-party copy is unverified. See `reports/source_provenance.md` and `reports/methodology_audit.md`.

Kaggle: create a notebook, import this file, enable Internet, choose T4 if available, and Run All. CPU mode is sufficient for the default audit. No credentials, paid API calls, model weights, or large corpus downloads occur by default. Optional sparse and checkpoint stages require the artifacts listed below. The Intent-Aware Supervision Guard and adversarial tests are absent.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, platform, random, importlib.util
REPO_URL = "https://github.com/Subhana2004/DualReform-Supervision-Audit.git"
AUDIT_COMMIT = "78127bf7c637fc8c6d35514134bfff632bf1236a"
KAGGLE = Path("/kaggle/working").is_dir()
ROOT = Path("/kaggle/working/DualReform-Supervision-Audit") if KAGGLE else Path.cwd()
if KAGGLE and not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", AUDIT_COMMIT], check=True)
if KAGGLE:
    actual = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
    if actual != AUDIT_COMMIT:
        raise RuntimeError("Existing checkout differs from pinned commit; use a fresh notebook session")
assert (ROOT / "configs/phase1_baseline.yaml").is_file(), "Run from the audit repository root"
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([str(ROOT / "src"), str(ROOT)]))
EXPORT = Path("/kaggle/working/phase1_exports") if KAGGLE else ROOT / "work/notebook_validation"
EXPORT.mkdir(parents=True, exist_ok=True)
print({"root": str(ROOT), "python": platform.python_version(), "pinned_commit": AUDIT_COMMIT})

## Dependencies and hardware
The default audit uses Python's standard library. Install the pinned audit/test environment only in Kaggle. Optional ML dependencies are a separate proposed compatibility profile, not a validated reproduction of the author's environment. No Torch replacement occurs unless the checkpoint stage is explicitly enabled.

In [ ]:
if KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-audit-lock.txt"], check=True)
def run_logged(command, name):
    result = subprocess.run(command, env=ENV, capture_output=True, text=True)
    (EXPORT / name).write_text(result.stdout + "\nSTDERR\n" + result.stderr, encoding="utf-8")
    print(result.stdout[-2500:])
    if result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}); inspect {name}: {result.stderr[-1500:]}")
    return result
print("Audit dependencies ready")

In [ ]:
random.seed(42)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
hardware = {"python": platform.python_version(), "cuda_available": False,
            "gpu_execution": "NOT EXECUTED", "gpus": []}
try:
    import torch
    torch.manual_seed(42)
    hardware.update(torch=torch.__version__, cuda=torch.version.cuda,
                    cuda_available=torch.cuda.is_available())
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
        hardware["gpus"] = [{"name": torch.cuda.get_device_name(i),
            "memory_bytes": torch.cuda.get_device_properties(i).total_memory,
            "capability": torch.cuda.get_device_capability(i)} for i in range(torch.cuda.device_count())]
except ImportError:
    hardware["torch"] = None
try:
    nvidia = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    (EXPORT / "nvidia-smi.txt").write_text(nvidia.stdout + nvidia.stderr)
except FileNotFoundError:
    (EXPORT / "nvidia-smi.txt").write_text("nvidia-smi unavailable; GPU not assumed")
from dualreform_audit.io import write_json
write_json(EXPORT / "hardware.json", hardware)
print(json.dumps(hardware, indent=2))

## Locked real data and smoke checks
TopiOCQA Zenodo v1 (CC BY 4.0) and QReCC at a pinned Apple commit (CC BY-SA 3.0). Downloads total about 16 MB. Data remain in ignored storage. TopiOCQA is the intended pipeline benchmark. QReCC context discrepancies are reported, and that dataset is not passed to the optional pseudo-reference runner. Training projections explicitly omit gold passages, rationales, and human rewrites.

In [ ]:
run_logged([sys.executable, "-m", "scripts.acquire_data"], "data_acquisition.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "tests.log")

In [ ]:
t0 = time.perf_counter()
run_logged([sys.executable, "-m", "scripts.run_data_audit", "--output", str(EXPORT / "data_integrity")],
           "data_integrity.log")
print("Data audit runtime seconds:", time.perf_counter() - t0)
print((EXPORT / "data_integrity/metrics.json").read_text()[:4000])

In [ ]:
# Offline smoke: real input projection and exact iteration-zero query construction.
from paper_reimplementation.signals import pseudo_query
sample_path = ROOT / "data/processed/sample_inputs.jsonl"
if sample_path.exists():
    sample = next(json.loads(s) for s in sample_path.read_text(encoding="utf-8").splitlines()
                  if json.loads(s)["dataset"] == "topiocqa" and json.loads(s)["split"] == "train")
    query = pseudo_query(sample["response"], iteration=0)
    assert query == sample["response"]
    print({"id": sample["conversation_id"], "history_turns": len(sample["history"]),
           "query_nonempty": bool(query.strip()), "model_loaded": False})
else:
    print("Run the data audit cell before the real-input smoke check")

## Optional paper-derived initial pseudo-reference retrieval
Attach the **full original TopiOCQA Lucene index** as a Kaggle Dataset, plus a corpus manifest documenting dataset, corpus version, index file checksum manifest, and document prefix. Do not build a corpus from gold passages. No index is attached here. This stage uses the original BM25 parameters and top-100 retrieval, and selects top 3 pseudo-references. It only reconstructs Algorithm 1 iteration 0; it does not produce refinement, candidate preferences, or a trained model.

If enabled, the profile pins `pyserini==0.25.0` and requires Java 11. Its Kaggle compatibility is unexecuted. Provision Java 11 if absent using Kaggle's package manager, then set JAVA_HOME before importing Pyserini. A modern preexisting Lucene index may be incompatible; rebuild from the documented original corpus using the matching backend.

In [ ]:
RUN_INITIAL_PSEUDO = False
INDEX_DIR = ""  # e.g. /kaggle/input/<your-index>/index
CORPUS_MANIFEST = ""  # JSON described in docs/kaggle.md
if RUN_INITIAL_PSEUDO:
    if not INDEX_DIR or not CORPUS_MANIFEST:
        raise RuntimeError("Attach the audited corpus/index and fill INDEX_DIR and CORPUS_MANIFEST")
    subprocess.run([sys.executable, "-m", "pip", "install", "pyserini==0.25.0", "numpy==1.26.4"], check=True)
    java = subprocess.run(["java", "-version"], capture_output=True, text=True)
    if java.returncode or 'version "11.' not in java.stderr:
        raise RuntimeError("This Pyserini profile requires Java 11; set JAVA_HOME before proceeding")
    from dualreform_audit.io import write_jsonl
    records = [json.loads(s) for s in sample_path.read_text(encoding="utf-8").splitlines()]
    train_inputs = EXPORT / "topiocqa_training_inputs.jsonl"
    write_jsonl(train_inputs, [r for r in records if r["dataset"] == "topiocqa" and r["split"] == "train"])
    run_logged([sys.executable, "-m", "scripts.run_initial_pseudo", "--inputs", str(train_inputs),
                "--index", INDEX_DIR, "--corpus-manifest", CORPUS_MANIFEST,
                "--output", str(EXPORT / "initial_pseudo"), "--limit", "32"], "initial_pseudo.log")
else:
    write_json(EXPORT / "initial_pseudo_status.json", {"status": "NOT EXECUTED", "blocker": "No original full corpus/index attached"})

## Optional trained checkpoint inference
No trained DualReform checkpoint was found. **Do not point this cell at base Llama and call it DualReform.** Attach a merged checkpoint, hash/license/revision provenance, and exact pre-rendered reviewed prompts. The runner preserves prompts and raw generated tokens. Single-T4 fp16 memory is insufficient under this conservative profile; at least 22 GiB aggregate GPU memory is required. Two T4s may fit with model sharding but have not been tested. fp16/eager and greedy decoding are explicit deviations; original bf16/FlashAttention settings are not assumed to work on T4. No automatic quantization, offload, fine-tuning, or candidate API use occurs.

Optional profile: PyTorch 2.5.1 CUDA 12.1, Transformers 4.43.0, Accelerate 0.33.0, NumPy 1.26.4. This differs from the source's PyTorch 2.1.2; run in a fresh session and record package versions. Full training remains blocked independently of inference.

In [ ]:
RUN_CHECKPOINT_INFERENCE = False
CHECKPOINT_DIR, CHECKPOINT_PROVENANCE, REVIEWED_PROMPTS = "", "", ""
if RUN_CHECKPOINT_INFERENCE:
    if not all([CHECKPOINT_DIR, CHECKPOINT_PROVENANCE, REVIEWED_PROMPTS]):
        raise RuntimeError("Acquire a trained checkpoint with provenance and reviewed prompts first")
    if not hardware["cuda_available"]:
        raise RuntimeError("Enable an available Kaggle GPU first")
    if sum(g["memory_bytes"] for g in hardware["gpus"]) < 22 * 1024**3:
        raise RuntimeError("One T4 does not meet the conservative fp16 8B memory budget")
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url",
                    "https://download.pytorch.org/whl/cu121"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "transformers==4.43.0",
                    "accelerate==0.33.0", "numpy==1.26.4"], check=True)
    run_logged([sys.executable, "-m", "scripts.run_checkpoint_inference", "--checkpoint", CHECKPOINT_DIR,
                "--provenance", CHECKPOINT_PROVENANCE, "--prompts", REVIEWED_PROMPTS,
                "--output", str(EXPORT / "checkpoint_inference")], "checkpoint_inference.log")
else:
    write_json(EXPORT / "checkpoint_status.json", {"status": "NOT EXECUTED", "blocker": "No verified trained DualReform checkpoint"})

## Optional retrieval evaluation and export
Evaluate a supplied standard six-column TREC run and dataset qrels in JSON (`{query_id: {passage_id: relevance}}`). Missing run queries count as zero; no-positive-qrel queries are explicitly excluded. Metrics are MRR over the supplied top-100 run, linear-gain nDCG@3, R@5 and R@20. Exact published eligibility still requires the original split/qrels mapping. The pure Python implementation has known-answer tests; optional `pytrec_eval` cross-checking below uses the official evaluator. No generation evaluation API is called.

In [ ]:
RUN_FILE, QRELS_FILE = "", ""
if RUN_FILE and QRELS_FILE:
    run_logged([sys.executable, "-m", "scripts.evaluate_run", "--run", RUN_FILE,
                "--qrels", QRELS_FILE, "--output", str(EXPORT / "evaluation.json")], "evaluation.log")
    # Official evaluator cross-check; this installation/profile is NOT locally executed.
    subprocess.run([sys.executable, "-m", "pip", "install", "pytrec-eval==0.5"], check=True)
    import pytrec_eval
    from dualreform_audit.metrics import read_trec
    qr = json.loads(Path(QRELS_FILE).read_text())
    runs = read_trec(RUN_FILE)
    scores = {q: {h["doc_id"]: h["score"] for h in hits} for q, hits in runs.items()}
    eligible = {q: v for q,v in qr.items() if any(g > 0 for g in v.values())}
    official = pytrec_eval.RelevanceEvaluator(eligible, {"recip_rank", "ndcg_cut.3", "recall.5", "recall.20"}).evaluate(scores)
    ours = json.loads((EXPORT / "evaluation.json").read_text())["per_example"]
    for q, values in official.items():
        for key, value in values.items():
            assert abs(value - ours[q][key]) < 1e-10, (q, key, value, ours[q][key])
    write_json(EXPORT / "official_evaluator_crosscheck.json", {"status": "passed", "queries": len(official)})
else:
    write_json(EXPORT / "evaluation_status.json", {"status": "NOT EXECUTED", "blocker": "No model run or mapped qrels supplied"})
run_logged([sys.executable, "-m", "pip", "freeze"], "pip_freeze.txt")
import shutil
archive_path = shutil.make_archive(str(EXPORT.parent / "phase1_exports"), "zip", root_dir=EXPORT)
print("Download results and logs from Kaggle Output:", archive_path)